# Q3 — Độ ẩm khuyến nghị (10–12%), lỗi hạt loại 1 và điểm cảm quan (`total_9`)

**Câu hỏi phân tích (bản chốt 08/11):**  
> *Các lô có độ ẩm trong khoảng khuyến nghị 10–12% và không có lỗi hạt loại 1 có `total_9` cao hơn các lô còn lại bao nhiêu điểm (CI 95%), và kết luận có giữ nguyên khi (a) giữ hay bỏ các lô `Moisture = 0`, (b) đổi ngưỡng outlier từ 1,5×IQR sang 3×IQR không?*

- **Người quan tâm:** Khâu phơi/sấy và bảo quản cà phê nhân xanh.
- **Phạm vi:** Phân tích chính trên các lô Arabica có độ ẩm dương (`Moisture > 0`); các lô Arabica có `Moisture = 0` được giữ riêng trong `df_arabica_full` để phục vụ kiểm tra độ bền (sensitivity/robustness check) ở bước D6.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.cleaning import convert_types, flag_placeholders, load_raw
from src.config import RANDOM_STATE, RAW, SCORE_COLS
from src.features import add_defect_total, add_total_score, bin_moisture


def major_minor(version: str) -> str:
    return ".".join(version.split(".")[:2])


# TEST: Kiểm tra phiên bản thư viện chuẩn của môi trường coffee
assert major_minor(np.__version__) == "1.26", f"numpy {np.__version__} khác environment.yml"
assert major_minor(pd.__version__) == "2.1", f"pandas {pd.__version__} khác environment.yml"
assert major_minor(matplotlib.__version__) == "3.8", f"matplotlib {matplotlib.__version__} khác environment.yml"
print("Environment OK:", f"numpy={np.__version__}, pandas={pd.__version__}, matplotlib={matplotlib.__version__}")

Environment OK: numpy=1.26.4, pandas=2.1.4, matplotlib=3.8.4


## 1. Preprocessing & Biến dẫn xuất (D3)

### Luồng xử lý dữ liệu
1. **Đọc dữ liệu & chuẩn hóa kiểu:** Dùng `load_raw()` (`keep_default_na=False, na_values=[""]` để giữ nguyên chuỗi `"None"` của cột `Color`), gọi `convert_types()` để chuyển `Expiration` sang `datetime` và `Harvest.Year` sang `Int64`, sau đó tính `total_9` bằng `add_total_score()`.
2. **Gắn cờ giá trị giữ chỗ:** Dùng `flag_placeholders()` để nhận diện `zero_all_scores` (bản ghi lỗi `REC_ID = 1310`) và `zero_moisture` (`Moisture == 0`).
3. **Lọc loài (`Species == "Arabica"`):** Loại bỏ 28 lô Robusta theo quyết định thống nhất của nhóm.
4. **Loại bỏ bản ghi lỗi `REC_ID = 1310`:** Loại 1 lô Arabica có toàn bộ 9 điểm cảm quan bằng 0 (`zero_all_scores == True`), tạo tập `df_arabica_full` (1.310 lô) để dùng cho kiểm tra độ bền (a) giữ `Moisture = 0`.
5. **Xử lý `Moisture = 0` theo quy tắc tuần 1:** Theo EDA tuần 1, `Moisture = 0` là giá trị giữ chỗ cho dữ liệu thiếu. Phân tích chính `df_q3` chỉ giữ các lô có độ ẩm dương (`Moisture > 0`).
6. **Tạo biến dẫn xuất (D3):**
   - `add_defect_total(df)`: tính `defects_total = Category.One.Defects + Category.Two.Defects + Quakers` để tổng hợp toàn bộ lỗi hạt.
   - `bin_moisture(df)`: phân nhóm `Moisture` thành `moisture_band` với 3 mức `<10%`, `10–12%`, `>12%` (`bins=(0, 0.10, 0.12, 1)`, `right=False`, tức `[0, 0.10)`, `[0.10, 0.12)`, `[0.12, 1)`).  
     - **Cơ sở trích nguồn (Justified D3):** Theo khuyến nghị kỹ thuật của **Specialty Coffee Association (SCA)** và tiêu chuẩn bảo quản cà phê nhân xanh (**ISO / hướng dẫn phơi sấy cà phê nhân xanh**), độ ẩm mục tiêu để bảo quản an toàn, giữ ổn định hương vị và hạn chế nấm mốc/lão hóa nằm trong khoảng **10–12%** (cà phê quá khô `<10%` dễ bạc màu, mất hương; quá ẩm `>12%` dễ phát sinh nấm mốc và giảm điểm chất lượng).

In [2]:
tracking = []

# Bước 1: Đọc dữ liệu thô, chuyển kiểu, tính total_9
df_raw = add_total_score(convert_types(load_raw()))
tracking.append(("1. Dữ liệu thô ban đầu", len(df_raw), 0, "Đọc Coffee_Qlty.csv + add_total_score"))
print(f"Bước 1 - Dữ liệu thô: {len(df_raw)} dòng")

# Bước 2: Gắn cờ placeholder
masks = flag_placeholders(df_raw)

# Bước 3: Lọc Arabica
df_arabica = df_raw[df_raw["Species"] == "Arabica"].copy()
tracking.append((
    "2. Lọc Species == 'Arabica'",
    len(df_arabica),
    len(df_raw) - len(df_arabica),
    "Loại các lô Robusta",
))
print(f"Bước 2 - Lọc Arabica: {len(df_arabica)} dòng (loại {len(df_raw) - len(df_arabica)})")

# Bước 4: Loại bản ghi lỗi REC_ID = 1310 (zero_all_scores)
df_arabica_full = df_arabica[~masks.loc[df_arabica.index, "zero_all_scores"]].copy()
tracking.append((
    "3. Loại placeholder điểm = 0 (REC_ID 1310)",
    len(df_arabica_full),
    len(df_arabica) - len(df_arabica_full),
    "Giữ làm tập sensitivity (có Moisture = 0)",
))
print(f"Bước 3 - Loại REC_ID 1310: {len(df_arabica_full)} dòng (loại {len(df_arabica) - len(df_arabica_full)})")

# Bước 5: Loại Moisture = 0 cho phân tích chính
df_q3 = df_arabica_full[~masks.loc[df_arabica_full.index, "zero_moisture"]].copy()
tracking.append((
    "4. Loại Moisture == 0 (phân tích chính)",
    len(df_q3),
    len(df_arabica_full) - len(df_q3),
    "Moisture = 0 là giá trị giữ chỗ (missing)",
))
print(f"Bước 4 - Tập phân tích chính (Moisture > 0): {len(df_q3)} dòng (loại {len(df_arabica_full) - len(df_q3)})")

# Bước 6: Tạo biến dẫn xuất (D3)
df_q3 = bin_moisture(add_defect_total(df_q3))
df_arabica_full = bin_moisture(add_defect_total(df_arabica_full))

tracking_df = pd.DataFrame(
    tracking, columns=["Bước xử lý", "Số dòng còn lại", "Số dòng bị loại", "Ghi chú"]
)
display(tracking_df)

print("\nPhân bố moisture_band trên tập phân tích chính (df_q3):")
display(df_q3["moisture_band"].value_counts(dropna=False).to_frame("count"))

print("\nBảng chéo moisture_band × không có lỗi hạt loại 1 (Category.One.Defects == 0):")
display(pd.crosstab(df_q3["moisture_band"], df_q3["Category.One.Defects"] == 0, margins=True))

Bước 1 - Dữ liệu thô: 1339 dòng
Bước 2 - Lọc Arabica: 1311 dòng (loại 28)
Bước 3 - Loại REC_ID 1310: 1310 dòng (loại 1)
Bước 4 - Tập phân tích chính (Moisture > 0): 1058 dòng (loại 252)


,Bước xử lý,Số dòng còn lại,Số dòng bị loại,Ghi chú
0,1. Dữ liệu thô ban đầu,1339,0,Đọc Coffee_Qlty.csv + add_total_score
1,2. Lọc Species == 'Arabica',1311,28,Loại các lô Robusta
2,3. Loại placeholder điểm = 0 (REC_ID 1310),1310,1,Giữ làm tập sensitivity (có Moisture = 0)
3,4. Loại Moisture == 0 (phân tích chính),1058,252,Moisture = 0 là giá trị giữ chỗ (missing)



Phân bố moisture_band trên tập phân tích chính (df_q3):


,count
moisture_band,
10–12%,561
>12%,405
<10%,92



Bảng chéo moisture_band × không có lỗi hạt loại 1 (Category.One.Defects == 0):


Category.One.Defects,False,True,All
moisture_band,,,
<10%,19,73,92
10–12%,67,494,561
>12%,74,331,405
All,160,898,1058


In [3]:
# TEST: Kiểm tra tính đúng đắn của preprocessing & biến dẫn xuất D3
assert (df_q3["Species"] == "Arabica").all(), "Phải chỉ còn Arabica"
assert (df_arabica_full["Species"] == "Arabica").all(), "df_arabica_full phải chỉ còn Arabica"
assert 1310 not in df_q3["REC_ID"].values, "REC_ID 1310 phải bị loại"
assert 1310 not in df_arabica_full["REC_ID"].values, "REC_ID 1310 phải bị loại khỏi df_arabica_full"
assert (df_q3["Moisture"] > 0).all(), "Tập chính df_q3 chỉ được chứa Moisture > 0"
assert df_q3["moisture_band"].isna().sum() == 0, "Không được có moisture_band bị NA trên tập Moisture > 0"
assert (df_q3["defects_total"] >= 0).all(), "defects_total không được âm"
assert (
    df_q3["defects_total"]
    == df_q3["Category.One.Defects"] + df_q3["Category.Two.Defects"] + df_q3["Quakers"]
).all(), "defects_total phải bằng tổng 3 cột lỗi"

# Kiểm tra tính bất biến (không sửa df gốc)
sample = df_raw.head(5)
cols_before = list(sample.columns)
_ = add_defect_total(sample)
_ = bin_moisture(sample)
assert list(sample.columns) == cols_before, "Hàm features không được sửa df gốc in-place"

print("All preprocessing & D3 # TEST checks passed!")

All preprocessing & D3 # TEST checks passed!


## 2. Analysis — Định lượng bất định (D5)

Thực hiện kiểm định phi tham số và ước lượng khoảng tin cậy bootstrap (95% CI, 5.000 lần lấy mẫu lại, `seed=RANDOM_STATE`) cho chênh lệch **median `total_9` (đơn vị: điểm)**:
1. **Theo nhóm độ ẩm (`moisture_band`):**
   - Kiểm định Kruskal–Wallis (`scipy.stats.kruskal`) trên cả 3 band (`<10%`, `10–12%`, `>12%`).
   - Dùng `bootstrap_diff` so sánh median `total_9` của nhóm khuyến nghị `10–12%` lần lượt với `<10%` và `>12%`.
2. **Theo mức độ lỗi hạt (`defects_total`):**
   - So sánh nhóm **không có lỗi (`defects_total == 0`)** với nhóm **có lỗi (`defects_total > 0`)** bằng `bootstrap_diff` (CI 95%) và kiểm định Kruskal–Wallis.
3. **Nhóm mục tiêu của câu hỏi Q3 (`10–12%` và `Category.One.Defects == 0` vs các lô còn lại):**
   - Định lượng mức chênh lệch median `total_9` (điểm, CI 95%) giữa các lô đạt đồng thời hai điều kiện (độ ẩm khuyến nghị `10–12%` và không có lỗi hạt loại 1) so với phần còn lại trên tập phân tích chính (`df_q3`).

In [4]:
from scipy.stats import kruskal
from src.stats import bootstrap_diff, cohens_d

d = df_q3.dropna(subset=["total_9", "moisture_band", "defects_total", "Category.One.Defects"]).copy()

# --- 2.1. Thống kê mô tả theo moisture_band ---
band_summary = (
    d.groupby("moisture_band", observed=True)["total_9"]
    .agg(
        n="count",
        median="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
        mean="mean",
        std="std",
    )
    .assign(IQR=lambda t: t["q75"] - t["q25"])
)
print("Thống kê total_9 theo moisture_band (điểm):")
display(band_summary[["n", "median", "IQR", "q25", "q75", "mean", "std"]].round(3))

# Kruskal-Wallis cho 3 band
groups_band = [g["total_9"].values for _, g in d.groupby("moisture_band", observed=True)]
kw_band_stat, kw_band_p = kruskal(*groups_band)
print(f"\nKruskal-Wallis (3 moisture_band): H = {kw_band_stat:.4f}, p = {kw_band_p:.4e}")

# Bootstrap diff: 10–12% vs <10% và 10–12% vs >12%
s_10_12 = d.loc[d["moisture_band"] == "10–12%", "total_9"]
s_lt_10 = d.loc[d["moisture_band"] == "<10%", "total_9"]
s_gt_12 = d.loc[d["moisture_band"] == ">12%", "total_9"]

diff_1012_lt10, ci_1012_lt10 = bootstrap_diff(s_10_12, s_lt_10)
diff_1012_gt12, ci_1012_gt12 = bootstrap_diff(s_10_12, s_gt_12)

# --- 2.2. So sánh ít lỗi (defects_total == 0) vs nhiều lỗi (defects_total > 0) ---
s_no_def = d.loc[d["defects_total"] == 0, "total_9"]
s_has_def = d.loc[d["defects_total"] > 0, "total_9"]
kw_def_stat, kw_def_p = kruskal(s_no_def.values, s_has_def.values)
diff_def, ci_def = bootstrap_diff(s_no_def, s_has_def)

# --- 2.3. So sánh nhóm mục tiêu Q3: (10–12% & Category.One.Defects == 0) vs còn lại ---
is_q3_target = (d["moisture_band"] == "10–12%") & (d["Category.One.Defects"] == 0)
s_q3_yes = d.loc[is_q3_target, "total_9"]
s_q3_no = d.loc[~is_q3_target, "total_9"]
kw_q3_stat, kw_q3_p = kruskal(s_q3_yes.values, s_q3_no.values)
diff_q3, ci_q3 = bootstrap_diff(s_q3_yes, s_q3_no)

# Tổng hợp bảng kết quả D5
d5_rows = [
    {
        "So sánh": "10–12% vs <10% (moisture_band)",
        "n_A": len(s_10_12),
        "n_B": len(s_lt_10),
        "median_A (điểm)": s_10_12.median(),
        "median_B (điểm)": s_lt_10.median(),
        "Chênh lệch median (điểm)": diff_1012_lt10,
        "CI 95% thấp (điểm)": ci_1012_lt10[0],
        "CI 95% cao (điểm)": ci_1012_lt10[1],
        "Kruskal p-value": kw_band_p,
        "Cohen's d": cohens_d(s_10_12, s_lt_10),
    },
    {
        "So sánh": "10–12% vs >12% (moisture_band)",
        "n_A": len(s_10_12),
        "n_B": len(s_gt_12),
        "median_A (điểm)": s_10_12.median(),
        "median_B (điểm)": s_gt_12.median(),
        "Chênh lệch median (điểm)": diff_1012_gt12,
        "CI 95% thấp (điểm)": ci_1012_gt12[0],
        "CI 95% cao (điểm)": ci_1012_gt12[1],
        "Kruskal p-value": kw_band_p,
        "Cohen's d": cohens_d(s_10_12, s_gt_12),
    },
    {
        "So sánh": "Ít lỗi (defects_total == 0) vs Có lỗi (> 0)",
        "n_A": len(s_no_def),
        "n_B": len(s_has_def),
        "median_A (điểm)": s_no_def.median(),
        "median_B (điểm)": s_has_def.median(),
        "Chênh lệch median (điểm)": diff_def,
        "CI 95% thấp (điểm)": ci_def[0],
        "CI 95% cao (điểm)": ci_def[1],
        "Kruskal p-value": kw_def_p,
        "Cohen's d": cohens_d(s_no_def, s_has_def),
    },
    {
        "So sánh": "Q3 mục tiêu (10–12% & Cat1 == 0) vs Còn lại",
        "n_A": len(s_q3_yes),
        "n_B": len(s_q3_no),
        "median_A (điểm)": s_q3_yes.median(),
        "median_B (điểm)": s_q3_no.median(),
        "Chênh lệch median (điểm)": diff_q3,
        "CI 95% thấp (điểm)": ci_q3[0],
        "CI 95% cao (điểm)": ci_q3[1],
        "Kruskal p-value": kw_q3_p,
        "Cohen's d": cohens_d(s_q3_yes, s_q3_no),
    },
]
d5_table = pd.DataFrame(d5_rows)
print("\nBảng tổng hợp định lượng bất định (D5):")
display(d5_table.round(4))

Thống kê total_9 theo moisture_band (điểm):


,n,median,IQR,q25,q75,mean,std
moisture_band,,,,,,,
<10%,92,75.455,1.93,74.49,76.42,75.525,1.928
10–12%,561,75.070,2.17,73.75,75.92,74.624,2.478
>12%,405,74.590,2.09,73.50,75.59,74.295,2.388



Kruskal-Wallis (3 moisture_band): H = 28.9730, p = 5.1120e-07



Bảng tổng hợp định lượng bất định (D5):


,So sánh,n_A,n_B,median_A (điểm),median_B (điểm),Chênh lệch median (điểm),CI 95% thấp (điểm),CI 95% cao (điểm),Kruskal p-value,Cohen's d
0,10–12% vs <10% (moisture_band),561,92,75.07,75.455,-0.385,-0.78,0.000,0.0000,-0.3741
1,10–12% vs >12% (moisture_band),561,405,75.07,74.590,0.480,0.24,0.660,0.0000,0.1347
2,Ít lỗi (defects_total == 0) vs Có lỗi (> 0),244,814,75.00,74.920,0.080,-0.16,0.240,0.3937,0.1700
3,Q3 mục tiêu (10–12% & Cat1 == 0) vs Còn lại,494,564,75.09,74.745,0.345,0.16,0.585,0.0014,0.1179


In [5]:
# Biểu đồ trực quan phân bố total_9 theo moisture_band và nhóm lỗi hạt
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

band_order = ["<10%", "10–12%", ">12%"]
band_data = [d.loc[d["moisture_band"] == b, "total_9"].values for b in band_order]
axes[0].boxplot(band_data, labels=band_order, showfliers=True)
axes[0].set_title(f"total_9 theo moisture_band (Kruskal p = {kw_band_p:.2e})")
axes[0].set_xlabel("Nhóm độ ẩm (moisture_band)")
axes[0].set_ylabel("total_9 (điểm)")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

def_data = [s_no_def.values, s_has_def.values]
axes[1].boxplot(def_data, labels=["defects_total == 0", "defects_total > 0"], showfliers=True)
axes[1].set_title(f"total_9 theo nhóm lỗi hạt (Kruskal p = {kw_def_p:.2e})")
axes[1].set_xlabel("Nhóm lỗi hạt")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

<Figure size 1200x450 with 2 Axes>

In [6]:
# TEST: Kiểm tra kết quả định lượng bất định D5
assert len(d5_table) == 4
assert not d5_table.isna().any().any(), "Không được có giá trị NaN trong bảng D5"
for _, row in d5_table.iterrows():
    assert row["CI 95% thấp (điểm)"] <= row["Chênh lệch median (điểm)"] <= row["CI 95% cao (điểm)"], (
        f"CI không bao phủ giá trị quan sát ở {row['So sánh']}"
    )
    assert 0 <= row["Kruskal p-value"] <= 1

print("All D5 Analysis # TEST checks passed!")

All D5 Analysis # TEST checks passed!
